# Experiment F — Synthetic Data Only (synthetic_only)
Train the segmentation U-Net on **synthetic data only**, with no real training data and no data augmentation transforms.

- **F**: ~1065 synthetic slices only, no real training data, no augmentation
- Val and test sets use real data as always
- Goal: measure standalone quality of DDPM-generated images — how well does the model generalize from synthetic to real data?

## 1. Colab Setup and GPU Check

In [ ]:
import os
import torch

COLAB_WORKING = "/content"
DRIVE_DIR = "/content/drive/MyDrive/mask-to-mri"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        gpu = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {gpu.name} ({gpu.total_memory/1e9:.1f} GB)")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("No GPU detected")

## 2. Mount Drive and Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = "/content/Mask-to-MRI"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/AmineAitLaamim/Mask-to-MRI.git
    print("Repository cloned")
else:
    %cd $REPO_DIR
    !git pull
    print("Repository updated")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

## 3. Prepare Required Folders

In [ ]:
LOCAL_DIRS = [
    "/content/Mask-to-MRI/dataset",
    "/content/Mask-to-MRI/notebooks",
]

DRIVE_DIRS = [
    f"{DRIVE_DIR}/dataset",
    f"{DRIVE_DIR}/experiment_B",
]

for path in LOCAL_DIRS + DRIVE_DIRS:
    os.makedirs(path, exist_ok=True)
    print(f"Ready: {path}")

## 4. Install Dependencies

In [ ]:
!pip install -q torch torchvision albumentations tifffile scikit-image pyyaml tqdm matplotlib opencv-python
print("Dependencies installed")

## 5. Experiment F Configuration

In [ ]:
EXPERIMENT_MODE = "synthetic_only"   # fixed — do not change
USE_AUGMENTATION = False              # always False — do not change
SYNTHETIC_ONLY = True                 # always True — do not change

print(f"Experiment mode: {EXPERIMENT_MODE}")
print(f"Augmentation: {USE_AUGMENTATION}")
print(f"Synthetic only: {SYNTHETIC_ONLY}")

## 6. Create Output Directories

In [ ]:
import os
for subfolder in ["checkpoints", "metrics", "plots", "samples"]:
    os.makedirs(
        f"/content/drive/MyDrive/mask-to-mri/experiment_B/synthetic_only/{subfolder}",
        exist_ok=True
    )
print("Output directories created")

## 7. Restore Dataset From Drive

In [ ]:
import shutil

RAW_DIR = f"{COLAB_WORKING}/Mask-to-MRI/dataset/lgg-mri-segmentation"
zip_in_drive = f"{DRIVE_DIR}/dataset/lgg-mri-segmentation.zip"
if os.path.exists(zip_in_drive):
    print(f"Found zip at: {zip_in_drive}")
    if os.path.exists(RAW_DIR):
        shutil.rmtree(RAW_DIR)
        print("Removed old extracted folder")
    shutil.unpack_archive(zip_in_drive, f"{COLAB_WORKING}/Mask-to-MRI/dataset")
    print("Dataset extracted to local Colab storage")
else:
    print("Dataset zip not found on Drive")
    print(f"Expected: {DRIVE_DIR}/dataset/lgg-mri-segmentation.zip")

print(f"RAW_DIR: {RAW_DIR}")

## 8. Restore Synthetic Data From Drive

In [ ]:
import glob

SYNTHETIC_DIR = f"{COLAB_WORKING}/Mask-to-MRI/dataset/synthetic_data"

# Experiment F requires synthetic data — always restore
synth_zip = f"{DRIVE_DIR}/dataset/synthetic_data.zip"
if os.path.exists(synth_zip):
    print(f"Found synthetic zip at: {synth_zip}")
    if os.path.exists(SYNTHETIC_DIR):
        shutil.rmtree(SYNTHETIC_DIR)
        print("Removed old extracted folder")
    shutil.unpack_archive(synth_zip, f"{COLAB_WORKING}/Mask-to-MRI/dataset")
    print("Synthetic data extracted to local Colab storage")
else:
    raise FileNotFoundError(
        f"Synthetic zip not found: {DRIVE_DIR}/dataset/synthetic_data.zip"
    )

synthetic_images = sorted(glob.glob(f"{SYNTHETIC_DIR}/*.png"))
synthetic_images = [p for p in synthetic_images if not p.endswith("_mask.png")]
synthetic_masks = sorted(glob.glob(f"{SYNTHETIC_DIR}/*_mask.png"))
print(f"Synthetic images: {len(synthetic_images)}")
print(f"Synthetic masks:  {len(synthetic_masks)}")
if len(synthetic_images) == 0 or len(synthetic_masks) == 0:
    raise RuntimeError(
        "Synthetic data folder is present but does not contain paired PNG files like synthetic_0001.png and synthetic_0001_mask.png"
    )

print(f"SYNTHETIC_DIR: {SYNTHETIC_DIR}")

## 9. Verify Synthetic Data Availability

In [ ]:
import os
syn_dir = "/content/Mask-to-MRI/dataset/synthetic_data"
n = len([f for f in os.listdir(syn_dir) if f.endswith(".png") and "_mask" not in f])
print(f"Synthetic images available: {n}")
assert n > 0, "No synthetic images found — cannot run experiment F"

## 10. Import Experiment B Package and Configure Paths

In [ ]:
import sys
for _mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[_mod]

sys.path.insert(0, '/content/Mask-to-MRI')

from src.experiment_B import CONFIG, build_experiment_b_dataloaders, create_unet, train_experiment_b, evaluate_experiment_b
from src.experiment_B import config as exp_b_config

# Inject experiment F parameters
exp_b_config.USE_AUGMENTATION = False
exp_b_config.SYNTHETIC_ONLY = True
exp_b_config.REAL_DATA_FRACTION = 1.0   # irrelevant but set for clarity
exp_b_config.SYNTHETIC_RATIO = 1        # irrelevant but set for clarity

# Route to the synthetic_only output folder
RUN_DIR = "MyDrive/mask-to-mri/experiment_B/synthetic_only"
CONFIG["baseline_run_name"] = "synthetic_only"
CONFIG["augmented_run_name"] = "synthetic_only"

OUTPUTS_DIR = f"{DRIVE_DIR}/experiment_B"
CONFIG["raw_dir"] = RAW_DIR
CONFIG["synthetic_dir"] = SYNTHETIC_DIR
CONFIG["outputs_base"] = OUTPUTS_DIR
CONFIG["drive_base"] = OUTPUTS_DIR
CONFIG["scheduler_t_max"] = CONFIG["epochs"]

print("Config loaded:")
print(f"  raw_dir: {CONFIG['raw_dir']}")
print(f"  synthetic_dir: {CONFIG['synthetic_dir']}")
print(f"  outputs_base: {CONFIG['outputs_base']}")
print(f"  drive_base: {CONFIG['drive_base']}")
print(f"  epochs: {CONFIG['epochs']}")
print(f"  batch_size: {CONFIG['batch_size']}")
print(f"  USE_AUGMENTATION: {exp_b_config.USE_AUGMENTATION}")
print(f"  SYNTHETIC_ONLY: {exp_b_config.SYNTHETIC_ONLY}")
print(f"  REAL_DATA_FRACTION: {exp_b_config.REAL_DATA_FRACTION}")
print(f"  SYNTHETIC_RATIO: {exp_b_config.SYNTHETIC_RATIO}")
print(f"  baseline_run_name: {CONFIG['baseline_run_name']}")
print(f"  augmented_run_name: {CONFIG['augmented_run_name']}")

## 11. Device Setup and Seeds

In [ ]:
import random
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

seed = CONFIG["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
print(f"Random seed fixed: {seed}")

## 12. Build DataLoaders

In [ ]:
loaders = build_experiment_b_dataloaders(CONFIG, mode="baseline")
for split, loader in loaders.items():
    print(f"{split}: {len(loader.dataset)} samples, {len(loader)} batches")

## 13. Verify Batch Shape

In [ ]:
images, masks = next(iter(loaders["train"]))
print(f"Image batch: {images.shape}")
print(f"Mask batch:  {masks.shape}")
print(f"Image mean: {images.mean():.3f}, std: {images.std():.3f}")
assert images.shape[1] == 1
assert masks.shape[1] == 1
print("Single-channel FLAIR segmentation input verified")

## 14. Model Sanity Check

In [ ]:
from src.experiment_B.losses import DiceBCELoss

model = create_unet(CONFIG).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {n_params:,} trainable parameters")

images_d = images[:2].to(device)
masks_d = masks[:2].to(device)
criterion = DiceBCELoss(CONFIG["loss_bce_weight"], CONFIG["loss_dice_weight"])
logits = model(images_d)
loss = criterion(logits, masks_d)
print(f"Forward pass OK - logits={logits.shape}, loss={loss.item():.4f}")

## 15. Restore Latest Checkpoint From Drive

In [ ]:
import glob
import shutil
from pathlib import Path

run_name = CONFIG["baseline_run_name"]
local_ckpt_dir = f"{OUTPUTS_DIR}/{run_name}/checkpoints"
drive_ckpt_dir = f"{DRIVE_DIR}/experiment_B/{run_name}/checkpoints"
os.makedirs(local_ckpt_dir, exist_ok=True)

drive_ckpts = glob.glob(f"{drive_ckpt_dir}/epoch_*.pt")
local_ckpts = glob.glob(f"{local_ckpt_dir}/epoch_*.pt")

if local_ckpts:
    print(f"Found {len(local_ckpts)} local checkpoint(s)")
elif drive_ckpts:
    latest = max(drive_ckpts, key=lambda p: int(Path(p).stem.split('_')[-1]))
    dest = os.path.join(local_ckpt_dir, os.path.basename(latest))
    if str(Path(latest).resolve()) != str(Path(dest).resolve()):
        shutil.copy2(latest, dest)
        print(f"Copied from Drive: {os.path.basename(latest)}")
    else:
        print(f"Drive checkpoint already at local path: {os.path.basename(latest)}")
else:
    print("No epoch checkpoints found")

local_ckpts = glob.glob(f"{local_ckpt_dir}/epoch_*.pt")
if local_ckpts:
    CONFIG["resume_from"] = max(local_ckpts, key=lambda p: int(Path(p).stem.split('_')[-1]))
    print(f"Resume checkpoint: {CONFIG['resume_from']}")
else:
    CONFIG["resume_from"] = None
    print("Starting from scratch")

## 16. Start Training

In [ ]:
train_result = train_experiment_b(CONFIG, mode="baseline")
print(train_result["best_checkpoint"])

## 17. Plot Training History

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = train_result["history_path"]
with open(history_path) as f:
    history = json.load(f)

epochs = [row["epoch"] for row in history]

plt.figure(figsize=(10, 4))
plt.plot(epochs, [row["train_loss"] for row in history], label="Train Loss")
plt.plot(epochs, [row["val_loss"] for row in history], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(epochs, [row["val_dice"] for row in history], label="Val Dice")
plt.plot(epochs, [row["val_iou"] for row in history], label="Val IoU")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 18. Evaluate Best Checkpoint on Test Set

In [ ]:
test_metrics = evaluate_experiment_b(CONFIG, train_result["best_checkpoint"], split="test")
print(json.dumps(test_metrics, indent=2))

## Summary

**Experiment F — Synthetic Data Only**

| Parameter | Value |
|-----------|-------|
| Training data | ~1065 synthetic slices only |
| Real training data | None (0 slices) |
| Augmentation | Off |
| Validation set | 151 real slices (unchanged) |
| Test set | 157 real slices (unchanged) |
| Output | `synthetic_only/` |

The model trained entirely on DDPM-generated synthetic data is evaluated on real held-out data. The val/test Dice score measures how well the DDPM captures real anatomical distribution.